In [ ]:
/**
 * ============================================================
 * CONSISTENT HASHING
 * ============================================================
 *
 * Distributed system mein ek common question:
 *
 *     "Ye particular key kis server par jayegi?"
 *
 *
 * Example:
 *
 *     User Profile
 *          |
 *          v
 *       user:42
 *
 *     Hume decide karna hai:
 *
 *       user:42 -> S0 ?
 *       user:42 -> S1 ?
 *       user:42 -> S2 ?
 *       user:42 -> S3 ?
 *
 *
 * Important requirement:
 *
 *     Same key ko normally same server par jaana chahiye.
 *
 *
 * Problem:
 *
 *     Agar server add/remove ho gaya,
 *     to bahut saari keys ka owner change
 *     nahi hona chahiye.
 *
 *
 * Consistent Hashing ka main goal:
 *
 *     Server change hone par
 *     minimum possible keys ko
 *     reassign karna.
 *
 * ============================================================
 * 1. SIMPLE EXAMPLE
 * ============================================================
 *
 * Suppose cache mein user profiles stored hain:
 *
 *
 *                 +------+
 *                 | S0   |
 *                 +------+
 *
 *                 +------+
 *                 | S1   |
 *                 +------+
 *
 *                 +------+
 *                 | S2   |
 *                 +------+
 *
 *                 +------+
 *                 | S3   |
 *                 +------+
 *
 *                 +------+
 *                 | S4   |
 *                 +------+
 *
 *
 * Request:
 *
 *     user:42
 *
 *
 * Hume consistently decide karna hai:
 *
 *     user:42 -> S2
 *
 *
 * Next time same request:
 *
 *     user:42 -> S2
 *
 *
 * Isse benefit:
 *
 *     Data already S2 par cached hai,
 *     therefore cache hit mil sakta hai.
 *
 *
 * ============================================================
 * 2. SIMPLE MODULO HASHING
 * ============================================================
 *
 * Sabse simple approach:
 *
 *
 *     serverIndex =
 *
 *         hash(key) % numberOfServers
 *
 *
 * Example:
 *
 *     Total servers = 5
 *
 *         S0
 *         S1
 *         S2
 *         S3
 *         S4
 *
 *
 * Key:
 *
 *     user:42
 *
 * Hash:
 *
 *     42
 *
 *
 * Then:
 *
 *     42 % 5
 *        |
 *        v
 *        2
 *
 *
 * Therefore:
 *
 *     user:42 -> S2
 *
 *
 * ============================================================
 * 3. MODULO HASHING EXAMPLE
 * ============================================================
 *
 *     Key Hash        hash % 5        Server
 *
 *       42              2              S2
 *       17              2              S2
 *       18              3              S3
 *       37              2              S2
 *       91              1              S1
 *
 *
 * Flow:
 *
 *     key
 *      |
 *      v
 *   hash(key)
 *      |
 *      v
 *     % 5
 *      |
 *      v
 *    Server
 *
 *
 * Advantage:
 *
 *     Simple
 *     Fast
 *     Same key -> same server
 *
 *
 * But ek BIG problem hai...
 *
 *
 * ============================================================
 * 4. PROBLEM WITH MODULO HASHING
 * ============================================================
 *
 * Suppose initially:
 *
 *     5 servers
 *
 *     S0 S1 S2 S3 S4
 *
 *
 * Formula:
 *
 *     hash(key) % 5
 *
 *
 * Ab ek new server add kar diya:
 *
 *     S0 S1 S2 S3 S4 S5
 *
 *
 * Formula becomes:
 *
 *     hash(key) % 6
 *
 *
 * Ab almost saari keys ka server change
 * ho sakta hai.
 *
 *
 * Example:
 *
 *
 *     Hash    5 Servers       6 Servers
 *
 *     42      42 % 5 = S2     42 % 6 = S0
 *     17      17 % 5 = S2     17 % 6 = S5
 *     18      18 % 5 = S3     18 % 6 = S0
 *     37      37 % 5 = S2     37 % 6 = S1
 *     91      91 % 5 = S1     91 % 6 = S1
 *
 *
 * Sirf 1 server add hua,
 * but 5 examples mein se 4 keys move ho gayi.
 *
 *
 * ============================================================
 * 5. REAL PROBLEM
 * ============================================================
 *
 * Imagine:
 *
 *     100 million cached keys
 *
 *     5 servers -> 6 servers
 *
 *
 * Modulo hashing ki wajah se
 * huge number of keys ka mapping change ho sakta hai.
 *
 *
 * Result:
 *
 *     Old cache:
 *
 *         S2 -> user data
 *
 *
 *     New request:
 *
 *         user -> S5
 *
 *
 *     But data S5 par nahi hai.
 *
 *         |
 *         v
 *      Cache Miss
 *         |
 *         v
 *      Database
 *         |
 *         v
 *      Fetch again
 *
 *
 * Isse:
 *
 *     Cache hit rate ↓
 *     Database load ↑
 *     Network traffic ↑
 *     System performance ↓
 *
 *
 * PDF ke example mein 5 se 6 servers karne par
 * 5 sample keys mein se 4 keys ka owner change hua.
 * :contentReference[oaicite:1]{index=1}
 *
 *
 * ============================================================
 * 6. SERVER REMOVE HONE PAR BHI SAME PROBLEM
 * ============================================================
 *
 * Suppose:
 *
 *     S4 server fail ho gaya.
 *
 *
 * Servers:
 *
 *     S0 S1 S2 S3
 *
 *
 * Formula:
 *
 *     hash(key) % 4
 *
 *
 * Again many keys ka owner change ho jayega.
 *
 *
 * Example:
 *
 *     Hash    Before        After
 *
 *     42      S2            S2
 *     17      S2            S1
 *     18      S3            S2
 *     37      S2            S1
 *     91      S1            S3
 *
 *
 * Goal should be:
 *
 *     Server remove hua
 *          |
 *          v
 *     Sirf us server ki keys move hon
 *
 * NOT:
 *
 *     Server remove hua
 *          |
 *          v
 *     Almost all keys reshuffle
 *
 *
 * ============================================================
 * 7. CONSISTENT HASHING
 * ============================================================
 *
 * Consistent Hashing ka solution:
 *
 *     Servers + Keys
 *
 * dono ko same fixed hash space mein place karte hain.
 *
 *
 * Is hash space ko:
 *
 *     HASH RING
 *
 * kehte hain.
 *
 *
 * Example:
 *
 *
 *                  S1
 *                  |
 *             +----+----+
 *           /             \
 *        S0                 S2
 *        |                   |
 *        |                   |
 *        S4                 S3
 *           \             /
 *             +---------+
 *
 *
 * Ring circular hota hai.
 *
 *
 * Example simplified range:
 *
 *     0 -> 999
 *
 *
 * After 999:
 *
 *     999 -> 0
 *
 *
 * Real systems mein hash space much larger
 * ho sakta hai:
 *
 *     0 -> 2^32 - 1
 *
 * or:
 *
 *     0 -> 2^64 - 1
 *
 *
 * ============================================================
 * 8. SERVER KO RING PAR PLACE KARNA
 * ============================================================
 *
 * Har server ke identifier ko hash karte hain.
 *
 *
 * Example:
 *
 *     S0 -> position 100
 *     S1 -> position 300
 *     S2 -> position 500
 *     S3 -> position 700
 *     S4 -> position 900
 *
 *
 * Ring:
 *
 *     0
 *     |
 *     v
 *    S0(100)
 *     |
 *     v
 *    S1(300)
 *     |
 *     v
 *    S2(500)
 *     |
 *     v
 *    S3(700)
 *     |
 *     v
 *    S4(900)
 *     |
 *     v
 *    999
 *     |
 *     +-------> back to 0
 *
 *
 * ============================================================
 * 9. KEY KO RING PAR PLACE KARNA
 * ============================================================
 *
 * Same process key ke saath:
 *
 *
 *     key
 *      |
 *      v
 *   hash(key)
 *      |
 *      v
 *   Ring position
 *
 *
 * Ab key ke position se
 * CLOCKWISE direction mein move karo.
 *
 *
 * Jo FIRST SERVER milega,
 * wahi key ka owner hoga.
 *
 *
 * Golden Rule:
 *
 *     Key -> First server clockwise
 *
 *
 * ============================================================
 * 10. KEY LOOKUP EXAMPLE
 * ============================================================
 *
 * Servers:
 *
 *     S0 = 100
 *     S1 = 300
 *     S2 = 500
 *     S3 = 700
 *     S4 = 900
 *
 *
 * KEY 1:
 *
 *     key position = 250
 *
 *
 * Clockwise:
 *
 *     250 -> S1(300)
 *
 *
 * Therefore:
 *
 *     250 -> S1
 *
 *
 * ------------------------------------------------------------
 *
 * KEY 2:
 *
 *     key position = 620
 *
 *
 * Clockwise:
 *
 *     620 -> S3(700)
 *
 *
 * Therefore:
 *
 *     620 -> S3
 *
 *
 * ------------------------------------------------------------
 *
 * KEY 3:
 *
 *     key position = 950
 *
 *
 * Ring end aa gaya:
 *
 *     950 -> 999 -> 0
 *
 *
 * Next server:
 *
 *     S0(100)
 *
 *
 * Therefore:
 *
 *     950 -> S0
 *
 *
 * This is called:
 *
 *     WRAP AROUND
 *
 *
 * ============================================================
 * 11. KEY EXACTLY SERVER POSITION PAR HO
 * ============================================================
 *
 * Example:
 *
 *     Key position = 700
 *
 *     S3 position = 700
 *
 *
 * Then:
 *
 *     Key -> S3
 *
 *
 * Rule:
 *
 *     Exact position match ho to
 *     wahi server owner hai.
 *
 *
 * ============================================================
 * 12. SERVER ADD KARNE KA ADVANTAGE
 * ============================================================
 *
 * Existing ring:
 *
 *     S0 = 100
 *     S1 = 300
 *     S2 = 500
 *     S3 = 700
 *     S4 = 900
 *
 *
 * New server:
 *
 *     S5 = 400
 *
 *
 * Before:
 *
 *     S1(300) -----> S2(500)
 *
 *
 * After:
 *
 *     S1(300) -> S5(400) -> S2(500)
 *
 *
 * S2 pehle:
 *
 *     300 -> 500
 *
 * range own karta tha.
 *
 *
 * Ab S5:
 *
 *     300 -> 400
 *
 * range own karega.
 *
 *
 * S2:
 *
 *     400 -> 500
 *
 * own karega.
 *
 *
 * Therefore sirf:
 *
 *     300 -> 400
 *
 * range ki keys move hongi.
 *
 * Baaki keys same server par rahengi.
 *
 *
 * ============================================================
 * 13. SERVER ADD EXAMPLE
 * ============================================================
 *
 * Key positions:
 *
 *     250
 *     350
 *     450
 *     620
 *     950
 *
 *
 * Before S5:
 *
 *     250 -> S1
 *     350 -> S2
 *     450 -> S2
 *     620 -> S3
 *     950 -> S0
 *
 *
 * S5 at 400 add hua.
 *
 *
 * After:
 *
 *     250 -> S1
 *     350 -> S5   <-- MOVED
 *     450 -> S2
 *     620 -> S3
 *     950 -> S0
 *
 *
 * Sirf 350 wali key move hui.
 *
 *
 * THIS IS THE MAIN ADVANTAGE
 * OF CONSISTENT HASHING.
 *
 *
 * ============================================================
 * 14. SERVER REMOVE KARNE KA ADVANTAGE
 * ============================================================
 *
 * Suppose:
 *
 *     S4 = 900
 *
 * fail ho gaya.
 *
 *
 * S4 ka range:
 *
 *     700 -> 900
 *
 *
 * S4 remove hone ke baad:
 *
 *     Keys clockwise next server par jayengi.
 *
 *
 * Next server:
 *
 *     S0 = 100
 *
 *
 * Example:
 *
 *     key = 850
 *
 *
 * Before:
 *
 *     850 -> S4
 *
 *
 * After S4 failure:
 *
 *     850 -> S0
 *
 *
 * But:
 *
 *     key = 620
 *
 * still:
 *
 *     620 -> S3
 *
 *
 * So only failed server ki range move hoti hai.
 *
 *
 * ============================================================
 * 15. KITNI KEYS MOVE HOTI HAIN?
 * ============================================================
 *
 * Agar:
 *
 *     N balanced servers
 *
 * hain aur 1 new server add karte hain,
 *
 * approximately:
 *
 *     1 / (N + 1)
 *
 * keys move hoti hain.
 *
 *
 * Example:
 *
 *     5 -> 6 servers
 *
 *
 * Approx:
 *
 *     1 / 6
 *
 *     = ~16.7%
 *
 *
 * IMPORTANT:
 *
 * Ye approximate average hai.
 *
 * Exact percentage depend karega:
 *
 *     Server positions
 *     Key distribution
 *     Hash distribution
 *
 *
 * ============================================================
 * 16. IMPORTANT: CONSISTENT HASHING
 *     DATA KO AUTOMATICALLY MOVE NAHI KARTA
 * ============================================================
 *
 * Ye interview mein important distinction hai.
 *
 *
 * Consistent Hashing:
 *
 *     Ownership / routing decide karta hai.
 *
 *
 * It does NOT automatically:
 *
 *     - Data transfer
 *     - Data copy
 *     - Replica repair
 *     - Durable write decision
 *
 *
 * Example:
 *
 *     S4 removed
 *
 * Consistent hashing says:
 *
 *     "Ab ye key S0 ki responsibility hai."
 *
 *
 * But actual data ko:
 *
 *     S4 -> S0
 *
 * transfer karna hai ya nahi,
 * woh separate mechanism ka responsibility hai.
 *
 *
 * Cache mein:
 *
 *     Data missing ho to database se
 *     dobara load kiya ja sakta hai.
 *
 *
 * Persistent storage mein:
 *
 *     Proper data migration required hai.
 *
 *
 * ============================================================
 * 17. PROBLEM: UNEVEN RANGES
 * ============================================================
 *
 * Real world mein server names ko hash karne par
 * positions perfectly evenly distribute nahi hoti.
 *
 *
 * Example:
 *
 *     S1 -> small range
 *     S2 -> huge range
 *
 *
 * Then:
 *
 *     S1 -> less keys
 *     S2 -> more keys
 *
 *
 * Result:
 *
 *     S2 par:
 *
 *         More traffic
 *         More memory
 *         More load
 *
 *
 * So simple one-position-per-server
 * perfect balancing guarantee nahi karta.
 *
 *
 * ============================================================
 * 18. SERVER FAILURE + UNEVEN RANGE
 * ============================================================
 *
 * Suppose:
 *
 *     S2 ka range bahut large hai.
 *
 *
 * S2 fails:
 *
 *     S2
 *      |
 *      X
 *
 *
 * Uska complete range next server ko mil jayega.
 *
 *
 * Result:
 *
 *     Neighbor server suddenly overloaded.
 *
 *
 * This is another problem.
 *
 *
 * Solution:
 *
 *     VIRTUAL NODES
 *
 *
 * ============================================================
 * 19. VIRTUAL NODES / VNODES
 * ============================================================
 *
 * Virtual node ka matlab:
 *
 *     Ek physical server ko ring par
 *     multiple positions dena.
 *
 *
 * Instead of:
 *
 *     S1
 *
 * We create:
 *
 *     S1-0
 *     S1-1
 *     S1-2
 *
 *
 * All of these point to:
 *
 *     Physical Server S1
 *
 *
 * Example:
 *
 *
 *     Physical Server
 *          |
 *          +---- S1-0 -> position 10
 *          |
 *          +---- S1-1 -> position 70
 *          |
 *          +---- S1-2 -> position 120
 *
 *
 * Similarly:
 *
 *     S2-0
 *     S2-1
 *     S2-2
 *
 *
 * ============================================================
 * 20. VNODE LOOKUP
 * ============================================================
 *
 * Suppose:
 *
 *     Key position = 75
 *
 *
 * Ring:
 *
 *     S1-0 = 10
 *     S3-0 = 30
 *     S2-0 = 50
 *     S1-1 = 70
 *     S2-1 = 80
 *     S3-1 = 90
 *     S1-2 = 120
 *
 *
 * Key:
 *
 *     75
 *
 *
 * First clockwise vnode:
 *
 *     S2-1 at 80
 *
 *
 * Therefore:
 *
 *     key 75 -> S2
 *
 *
 * Notice:
 *
 *     S2-1 is virtual node
 *
 *     But actual request:
 *
 *         -> Physical Server S2
 *
 *
 * ============================================================
 * 21. WHY VIRTUAL NODES HELP?
 * ============================================================
 *
 * WITHOUT VIRTUAL NODES:
 *
 *     One server
 *         |
 *         v
 *     One large range
 *
 *
 * WITH VIRTUAL NODES:
 *
 *     One server
 *       |
 *       +--> small range
 *       +--> small range
 *       +--> small range
 *       +--> small range
 *
 *
 * Benefits:
 *
 *     1. Better load distribution
 *
 *     2. Failure load multiple servers mein spread
 *
 *     3. Different server capacities ko handle kar sakte hain
 *
 *
 * ============================================================
 * 22. FAILURE WITH VIRTUAL NODES
 * ============================================================
 *
 * Suppose S2 has:
 *
 *     S2-0
 *     S2-1
 *     S2-2
 *
 *
 * S2 fails.
 *
 *
 * Instead of:
 *
 *     Entire S2 range
 *          |
 *          v
 *     One neighbor
 *
 *
 * We get:
 *
 *     S2-0 range
 *          |
 *          v
 *         S1
 *
 *     S2-1 range
 *          |
 *          v
 *         S3
 *
 *     S2-2 range
 *          |
 *          v
 *         S1
 *
 *
 * Load multiple servers mein spread ho jata hai.
 *
 *
 * ============================================================
 * 23. DIFFERENT SERVER CAPACITY
 * ============================================================
 *
 * Suppose:
 *
 *     S1 = weak server
 *
 *     S2 = powerful server
 *
 *
 * We can assign:
 *
 *     S1 -> 50 virtual nodes
 *
 *     S2 -> 100 virtual nodes
 *
 *
 * Therefore S2 ko:
 *
 *     More expected share
 *
 * milega.
 *
 *
 * This is called:
 *
 *     Capacity weighting
 *
 *
 * ============================================================
 * 24. VIRTUAL NODE TRADE-OFF
 * ============================================================
 *
 * More virtual nodes:
 *
 *     Better distribution
 *
 *     Better failure spreading
 *
 *     Better capacity weighting
 *
 *
 * BUT:
 *
 *     More metadata
 *     More routing information
 *     Membership changes par more work
 *
 *
 * Therefore:
 *
 *     Infinite vnodes nahi banate.
 *
 *
 * Practical approach:
 *
 *     Enough vnodes
 *     + measure actual distribution
 *
 *
 * PDF ke according tens se few hundred virtual nodes
 * per server ek common starting point ho sakta hai,
 * lekin exact number system ke distribution ko measure
 * karke choose karna chahiye. :contentReference[oaicite:2]{index=2}
 *
 *
 * ============================================================
 * 25. CONSISTENT HASH RING IMPLEMENTATION
 * ============================================================
 *
 * Actual code mein ring ko circular data structure
 * banana zaroori nahi hai.
 *
 *
 * We can maintain:
 *
 *     Sorted Array:
 *
 *         [10, 30, 50, 70, 80, 90, 120]
 *
 *
 * And mapping:
 *
 *     position -> physical server
 *
 *
 * Example:
 *
 *     10  -> S1
 *     30  -> S3
 *     50  -> S2
 *     70  -> S1
 *     80  -> S2
 *     90  -> S3
 *     120 -> S1
 *
 *
 * ============================================================
 * 26. KEY LOOKUP USING BINARY SEARCH
 * ============================================================
 *
 * Suppose:
 *
 *     keyHash = 75
 *
 *
 * Sorted positions:
 *
 *     [10, 30, 50, 70, 80, 90, 120]
 *
 *
 * Find first position >= 75.
 *
 *
 * Result:
 *
 *     80
 *
 *
 * Therefore:
 *
 *     75 -> server at position 80
 *
 *
 * This can be done using:
 *
 *     Binary Search
 *
 *
 * Complexity:
 *
 *     O(log V)
 *
 *
 * where:
 *
 *     V = number of virtual nodes
 *
 *
 * ============================================================
 * 27. WRAP AROUND IN IMPLEMENTATION
 * ============================================================
 *
 * Suppose:
 *
 *     keyHash = 150
 *
 *
 * But largest position:
 *
 *     120
 *
 *
 * No position >= 150.
 *
 *
 * Therefore:
 *
 *     Wrap around
 *
 *
 * Use first position:
 *
 *     10
 *
 *
 * Therefore:
 *
 *     150 -> server at position 10
 *
 *
 * ============================================================
 * 28. EMPTY RING
 * ============================================================
 *
 * Agar:
 *
 *     Ring empty hai
 *
 *
 * Then:
 *
 *     No server available.
 *
 *
 * Return:
 *
 *     Error
 *
 *
 * ============================================================
 * 29. HASH COLLISION
 * ============================================================
 *
 * Kabhi-kabhi two virtual nodes
 * same hash position par aa sakte hain.
 *
 *
 * Example:
 *
 *     S1-2 -> 500
 *
 *     S2-4 -> 500
 *
 *
 * Production implementation mein
 * deterministic rule hona chahiye.
 *
 *
 * Example:
 *
 *     Server ID sorting ke basis par
 *     ek fixed winner choose karna.
 *
 *
 * Goal:
 *
 *     Every client same result dekhe.
 *
 *
 * ============================================================
 * 30. HASH FUNCTION
 * ============================================================
 *
 * Production ke liye:
 *
 *     Fast
 *     Stable
 *     Well-distributed
 *     Non-cryptographic hash
 *
 * use kiya ja sakta hai.
 *
 *
 * Examples mentioned in the material:
 *
 *     MurmurHash
 *     xxHash
 *     CityHash
 *
 *
 * Example implementation mein:
 *
 *     FNV-1a
 *
 * use kiya gaya hai because it is simple
 * and gives consistent output across languages.
 *
 *
 * Important:
 *
 *     Hash function ka output stable hona chahiye.
 *
 *
 * ============================================================
 * 31. SAME HASH FUNCTION + SAME SERVER LIST
 * ============================================================
 *
 * Consistent hashing ka ek important requirement:
 *
 *
 * Every client should have:
 *
 *     Same server list
 *
 *     +
 *
 *     Same hash function
 *
 *
 * Otherwise:
 *
 *     Client A:
 *
 *         user:42 -> S2
 *
 *
 *     Client B:
 *
 *         user:42 -> S3
 *
 *
 * Then consistency break ho jayegi.
 *
 *
 * Usually:
 *
 *     Configuration service
 *
 * ring updates publish kar sakti hai
 * so all clients eventually same view
 * par converge karein.
 *
 *
 * ============================================================
 * 32. REPLICATION WITH CONSISTENT HASHING
 * ============================================================
 *
 * Consistent hashing ke saath replication bhi
 * implement ki ja sakti hai.
 *
 *
 * Suppose:
 *
 *     3 copies chahiye.
 *
 *
 * First:
 *
 *     Primary owner
 *
 *
 * Then clockwise move karo:
 *
 *     Next DISTINCT physical server
 *
 *
 * Example:
 *
 *     Key = 75
 *
 *
 * First:
 *
 *     position 80
 *     S2
 *
 *     => Primary
 *
 *
 * Next:
 *
 *     position 90
 *     S3
 *
 *     => Replica 1
 *
 *
 * Next:
 *
 *     position 120
 *     S1
 *
 *     => Replica 2
 *
 *
 * Final:
 *
 *     S2 -> Primary
 *     S3 -> Replica
 *     S1 -> Replica
 *
 *
 * ============================================================
 * 33. WHY "DISTINCT PHYSICAL SERVER" IMPORTANT?
 * ============================================================
 *
 * Suppose:
 *
 *     Primary:
 *
 *         S2-1
 *
 *
 * Next vnode:
 *
 *         S2-2
 *
 *
 * Dono same physical machine S2 ke hain.
 *
 *
 * If S2 crashes:
 *
 *     Primary + replica
 *
 * dono lost.
 *
 *
 * Therefore:
 *
 *     Replica select karte time
 *     physical server distinct hona chahiye.
 *
 *
 * Production mein replicas ko:
 *
 *     Different servers
 *     Different racks
 *     Different availability zones
 *
 * mein spread karna useful ho sakta hai.
 *
 *
 * ============================================================
 * 34. IMPORTANT: RING DOES NOT HANDLE REPLICATION ITSELF
 * ============================================================
 *
 * Ring sirf decide karta hai:
 *
 *     Primary kahan?
 *
 *     Replica candidates kahan?
 *
 *
 * Other components handle:
 *
 *     Data copying
 *     Write success rules
 *     Replica repair
 *     Missing replica recovery
 *
 *
 * ============================================================
 * 35. WHEN TO USE CONSISTENT HASHING?
 * ============================================================
 *
 * Good use cases:
 *
 *     1. Distributed Cache
 *
 *     2. Storage Sharding
 *
 *     3. Stateful Routing
 *
 *
 * Example:
 *
 *     userId -> same cache server
 *
 *
 *     productId -> same shard
 *
 *
 *     sessionId -> same server
 *
 *
 * ============================================================
 * 36. WHEN NOT TO USE IT?
 * ============================================================
 *
 * Agar:
 *
 *     Any server can handle any request
 *
 * then normal load balancer simpler ho sakta hai.
 *
 *
 * Example:
 *
 *
 *     Client
 *        |
 *        v
 *       LB
 *      / | \
 *     /  |  \
 *    S1 S2  S3
 *
 *
 * Agar servers stateless hain,
 * request kisi bhi healthy server ko ja sakti hai.
 *
 *
 * In that case consistent hashing ki
 * complexity unnecessary ho sakti hai.
 *
 *
 * ============================================================
 * 37. CONSISTENT HASHING vs LOAD BALANCING
 * ============================================================
 *
 * CONSISTENT HASHING:
 *
 *     Key-based routing
 *
 *     Same key usually
 *     same owner.
 *
 *
 * LOAD BALANCER:
 *
 *     Healthy server choose karta hai.
 *
 *     Any server request handle kar sakta hai.
 *
 *
 * Simple rule:
 *
 *     Stateful / key affinity
 *         |
 *         v
 *     Consistent Hashing
 *
 *
 *     Stateless service
 *         |
 *         v
 *     Normal Load Balancer
 *
 *
 * ============================================================
 * 38. OTHER APPROACHES
 * ============================================================
 *
 * Consistent hashing hi only solution nahi hai.
 *
 *
 * 1. RENDEZVOUS HASHING
 *
 *     Har server ko key ke liye score do.
 *
 *     Highest score wala server
 *     owner banega.
 *
 *
 * Good for:
 *
 *     Small / medium server sets
 *
 *
 * ------------------------------------------------------------
 *
 * 2. FIXED PARTITIONS
 *
 *     Keys ko fixed number of partitions mein divide karo.
 *
 *
 *     Example:
 *
 *         64 partitions
 *
 *
 *     key
 *      |
 *      v
 *   hash(key) % 64
 *      |
 *      v
 *   Partition 17
 *      |
 *      v
 *      S3
 *
 *
 * Partition count fixed hai.
 *
 * Server mapping change ho sakti hai,
 * but partition count same rehta hai.
 *
 *
 * Good for:
 *
 *     Logs
 *     Queues
 *     Databases
 *
 *
 * ============================================================
 * 39. CONSISTENT HASHING vs MODULO HASHING
 * ============================================================
 *
 *
 * MODULO:
 *
 *     server = hash(key) % N
 *
 *
 * Problem:
 *
 *     N change hua
 *         |
 *         v
 *     Many keys remapped
 *
 *
 * CONSISTENT HASHING:
 *
 *     Key + Servers -> Hash Ring
 *
 *
 *     Server add/remove
 *          |
 *          v
 *     Only nearby range affected
 *
 *
 * ============================================================
 * 40. INTERVIEW QUESTIONS
 * ============================================================
 *
 *
 * Q1. What is consistent hashing?
 *
 * Answer:
 *
 *     Consistent hashing ek technique hai
 *     jo keys ko servers ke beech map karti hai
 *     while minimizing key movement when
 *     servers join or leave the cluster.
 *
 *
 * ------------------------------------------------------------
 *
 * Q2. Why do we need consistent hashing?
 *
 * Answer:
 *
 *     Normal modulo hashing mein server count
 *     change hone par large number of keys
 *     remap ho jati hain.
 *
 *     Consistent hashing mein sirf affected
 *     range ki keys generally move hoti hain.
 *
 *
 * ------------------------------------------------------------
 *
 * Q3. What is the problem with:
 *
 *     hash(key) % numberOfServers ?
 *
 * Answer:
 *
 *     Server count change hote hi modulo value
 *     change ho jati hai.
 *
 *     Isliye large number of keys ka owner change
 *     ho sakta hai.
 *
 *
 * ------------------------------------------------------------
 *
 * Q4. What is a hash ring?
 *
 * Answer:
 *
 *     Hash ring ek fixed circular hash space hai
 *     jahan both servers aur keys ko hash karke
 *     positions assign ki jati hain.
 *
 *     Key ka owner first server clockwise
 *     hota hai.
 *
 *
 * ------------------------------------------------------------
 *
 * Q5. How is a key mapped to a server?
 *
 * Answer:
 *
 *     1. Key ko hash karo.
 *     2. Hash ko ring par position karo.
 *     3. Clockwise move karo.
 *     4. First server position ko owner banao.
 *
 *
 * ------------------------------------------------------------
 *
 * Q6. What happens when a new server joins?
 *
 * Answer:
 *
 *     New server ring par ek ya multiple positions
 *     leta hai.
 *
 *     Sirf uski newly acquired range ki keys
 *     previous owner se new server par remap hoti hain.
 *
 *
 * ------------------------------------------------------------
 *
 * Q7. What happens when a server leaves?
 *
 * Answer:
 *
 *     Us server ki owned keys clockwise
 *     next available servers ko remap hoti hain.
 *
 *     Baaki keys normally unaffected rehti hain.
 *
 *
 * ------------------------------------------------------------
 *
 * Q8. What are virtual nodes?
 *
 * Answer:
 *
 *     Virtual nodes ek physical server ko ring par
 *     multiple positions dete hain.
 *
 *     Example:
 *
 *         S1
 *         |
 *         +-- S1-0
 *         +-- S1-1
 *         +-- S1-2
 *
 *
 *     Ye load distribution improve karte hain.
 *
 *
 * ------------------------------------------------------------
 *
 * Q9. Why are virtual nodes needed?
 *
 * Answer:
 *
 *     Without vnodes, server positions uneven ho
 *     sakti hain.
 *
 *     Ek server bahut large range own kar sakta hai.
 *
 *     Vnodes multiple smaller ranges create karke
 *     load ko better distribute karte hain.
 *
 *
 * ------------------------------------------------------------
 *
 * Q10. What happens when a vnode server fails?
 *
 * Answer:
 *
 *     Us server ke multiple ranges different
 *     neighboring servers ke paas distribute
 *     ho sakte hain.
 *
 *     Isliye ek single neighbor par huge load
 *     nahi aata.
 *
 *
 * ------------------------------------------------------------
 *
 * Q11. Can consistent hashing guarantee
 *     perfectly equal distribution?
 *
 * Answer:
 *
 *     No.
 *
 *     Virtual nodes distribution ko statistically
 *     improve karte hain, but perfect equality
 *     guarantee nahi hoti.
 *
 *
 * ------------------------------------------------------------
 *
 * Q12. Does consistent hashing move data?
 *
 * Answer:
 *
 *     No.
 *
 *     Consistent hashing mainly ownership/routing
 *     decide karta hai.
 *
 *     Actual data migration separate process hai.
 *
 *
 * ------------------------------------------------------------
 *
 * Q13. How does consistent hashing support replication?
 *
 * Answer:
 *
 *     Primary owner find karne ke baad
 *     clockwise continue karke additional
 *     DISTINCT physical servers choose kiye
 *     ja sakte hain as replicas.
 *
 *
 * ------------------------------------------------------------
 *
 * Q14. Why should replicas be on different
 *     physical servers?
 *
 * Answer:
 *
 *     Agar primary aur replica same machine par
 *     hain aur machine fail ho gayi,
 *     dono copies lost ho sakti hain.
 *
 *
 * ------------------------------------------------------------
 *
 * Q15. When would you prefer a normal load balancer?
 *
 * Answer:
 *
 *     Jab service stateless ho aur koi bhi
 *     healthy server request handle kar sake.
 *
 *     Aise case mein consistent hashing ki
 *     complexity unnecessary ho sakti hai.
 *
 *
 * ------------------------------------------------------------
 *
 * Q16. What is the lookup complexity?
 *
 * Answer:
 *
 *     Sorted virtual-node positions ke saath
 *     binary search use kar sakte hain.
 *
 *
 *         Lookup = O(log V)
 *
 *
 *     where V = number of virtual nodes.
 *
 *
 * ------------------------------------------------------------
 *
 * Q17. What must all clients agree on?
 *
 * Answer:
 *
 *     All clients should use:
 *
 *         Same hash function
 *         Same server/ring membership
 *
 *     Otherwise same key different servers
 *     par route ho sakti hai.
 *
 *
 * ============================================================
 * 41. REAL SYSTEM DESIGN EXAMPLE
 * ============================================================
 *
 * Suppose:
 *
 *     Distributed Redis-like cache
 *
 * Servers:
 *
 *     S1
 *     S2
 *     S3
 *
 *
 * User profile:
 *
 *     user:1001
 *
 *
 * Flow:
 *
 *
 *     Request
 *        |
 *        v
 *     hash(user:1001)
 *        |
 *        v
 *     Hash Ring
 *        |
 *        v
 *     First Server Clockwise
 *        |
 *        v
 *       S2
 *        |
 *        v
 *     Cache lookup
 *
 *
 * Same user request:
 *
 *     user:1001
 *        |
 *        v
 *       S2
 *
 *
 * Therefore cache locality maintained.
 *
 *
 * ============================================================
 * 42. IF S4 IS ADDED
 * ============================================================
 *
 * Existing:
 *
 *     S1
 *     S2
 *     S3
 *
 *
 * Add:
 *
 *     S4
 *
 *
 * Ring changes only around S4's positions.
 *
 *
 * Therefore:
 *
 *     Most existing keys
 *     remain on same servers.
 *
 *
 * Benefit:
 *
 *     Cache hit rate better
 *     Less data reshuffling
 *     Less load spike
 *
 *
 * ============================================================
 * 43. IF S2 FAILS
 * ============================================================
 *
 *     S2
 *      |
 *      X
 *
 *
 * Consistent hashing identifies:
 *
 *     S2 ki ranges
 *
 *
 * Those ranges:
 *
 *     -> Next available owners
 *
 *
 * With virtual nodes:
 *
 *     Load multiple servers mein spread
 *     ho sakta hai.
 *
 *
 * ============================================================
 * 44. MOST IMPORTANT THINGS TO REMEMBER
 * ============================================================
 *
 *
 * 1.
 *
 *     Modulo hashing:
 *
 *         hash(key) % N
 *
 *
 *     N changes -> many keys move.
 *
 *
 * ------------------------------------------------------------
 *
 * 2.
 *
 *     Consistent hashing:
 *
 *         Key + Servers -> Hash Ring
 *
 *
 * ------------------------------------------------------------
 *
 * 3.
 *
 *     Key owner:
 *
 *         First server clockwise.
 *
 *
 * ------------------------------------------------------------
 *
 * 4.
 *
 *     Server added:
 *
 *         Only affected range moves.
 *
 *
 * ------------------------------------------------------------
 *
 * 5.
 *
 *     Server removed:
 *
 *         Only its owned range moves.
 *
 *
 * ------------------------------------------------------------
 *
 * 6.
 *
 *     Virtual nodes:
 *
 *         Better distribution
 *         Better failure handling
 *         Capacity weighting
 *
 *
 * ------------------------------------------------------------
 *
 * 7.
 *
 *     Consistent hashing:
 *
 *         Decides placement.
 *
 *     It does NOT automatically:
 *
 *         Move data
 *         Copy replicas
 *         Repair replicas
 *         Decide durable writes
 *
 *
 * ============================================================
 * 45. EASY MEMORY TRICK
 * ============================================================
 *
 *
 * MODULO HASHING:
 *
 *     "Server count change karo,
 *      mapping almost completely change ho sakti hai."
 *
 *
 * CONSISTENT HASHING:
 *
 *     "Ring par nearby ownership change karo."
 *
 *
 * VIRTUAL NODES:
 *
 *     "Ek server ko ring par
 *      multiple small positions do."
 *
 *
 * ============================================================
 * ONE-LINE INTERVIEW ANSWER
 * ============================================================
 *
 * "Consistent hashing maps both keys and servers onto
 * a fixed hash ring and assigns each key to the first
 * server clockwise, so when servers are added or removed,
 * only a small portion of keys need to be remapped instead
 * of reshuffling the entire cluster."
 *
 * ============================================================
 */

In [ ]:
/**
 * CONSISTENT HASHING
 *
 * Distributed system mein ek common question:
 *   "Ye particular key kis server par jayegi?"
 *
 * Example: user:42 -> S0? S1? S2? S3?
 *
 * Requirement : same key ko normally same server par jaana chahiye.
 * Problem     : server add/remove hone par bahut saari keys ka owner
 *               change nahi hona chahiye.
 *
 * Main goal: server change hone par minimum possible keys ko reassign karna.
 */

/**
 * 1. SIMPLE EXAMPLE
 *    Cache mein user profiles stored hain, servers: S0, S1, S2, S3, S4.
 *
 *    Request: user:42 -> consistently S2.
 *    Next time same request: user:42 -> S2.
 *
 *    Benefit: data already S2 par cached hai, so cache hit mil sakta hai.
 */

/**
 * 2. SIMPLE MODULO HASHING
 *    serverIndex = hash(key) % numberOfServers
 *
 *    Example: 5 servers (S0..S4), key user:42, hash = 42
 *      42 % 5 = 2 -> user:42 -> S2
 */

/**
 * 3. MODULO HASHING EXAMPLE
 *
 *      Key Hash    hash % 5    Server
 *        42           2          S2
 *        17           2          S2
 *        18           3          S3
 *        37           2          S2
 *        91           1          S1
 *
 *    Flow: key -> hash(key) -> % 5 -> Server
 *
 *    Advantage: simple, fast, same key -> same server.
 *    But ek BIG problem hai...
 */

/**
 * 4. PROBLEM WITH MODULO HASHING
 *    Initially 5 servers (S0..S4): hash(key) % 5.
 *    Ek new server add kiya (S0..S5): hash(key) % 6.
 *    Ab almost saari keys ka server change ho sakta hai.
 *
 *      Hash    5 Servers       6 Servers
 *      42      42 % 5 = S2     42 % 6 = S0
 *      17      17 % 5 = S2     17 % 6 = S5
 *      18      18 % 5 = S3     18 % 6 = S0
 *      37      37 % 5 = S2     37 % 6 = S1
 *      91      91 % 5 = S1     91 % 6 = S1
 *
 *    Sirf 1 server add hua, but 5 examples mein se 4 keys move ho gayi.
 */

/**
 * 5. REAL PROBLEM
 *    Imagine 100 million cached keys, 5 servers -> 6 servers.
 *    Modulo hashing ki wajah se huge number of keys ka mapping change ho sakta hai.
 *
 *    Old cache : S2 -> user data
 *    New request: user -> S5, but data S5 par nahi hai
 *      -> Cache Miss -> Database -> Fetch again
 *
 *    Result:
 *      Cache hit rate down, Database load up,
 *      Network traffic up, System performance down.
 */

/**
 * 6. SERVER REMOVE HONE PAR BHI SAME PROBLEM
 *    S4 fail ho gaya -> servers S0..S3 -> hash(key) % 4
 *    Again many keys ka owner change ho jayega.
 *
 *      Hash    Before    After
 *      42      S2        S2
 *      17      S2        S1
 *      18      S3        S2
 *      37      S2        S1
 *      91      S1        S3
 *
 *    Goal should be: server remove hua -> sirf us server ki keys move hon.
 *    NOT: server remove hua -> almost all keys reshuffle.
 */

/**
 * 7. CONSISTENT HASHING
 *    Solution: Servers + Keys dono ko same fixed hash space mein place karo.
 *    Is hash space ko HASH RING kehte hain.
 *
 *                   S1
 *               /       \
 *            S0           S2
 *             |            |
 *            S4           S3
 *               \       /
 *
 *    Ring circular hota hai.
 *    Simplified range: 0 -> 999, aur 999 ke baad wapas 0.
 *    Real systems mein hash space bada hota hai: 0 -> 2^32 - 1 or 0 -> 2^64 - 1.
 */

/**
 * 8. SERVER KO RING PAR PLACE KARNA
 *    Har server ke identifier ko hash karte hain.
 *
 *      S0 -> 100,  S1 -> 300,  S2 -> 500,  S3 -> 700,  S4 -> 900
 *
 *    Ring: 0 -> S0(100) -> S1(300) -> S2(500) -> S3(700) -> S4(900)
 *          -> 999 -> back to 0
 */

/**
 * 9. KEY KO RING PAR PLACE KARNA
 *    key -> hash(key) -> Ring position.
 *    Ab key ke position se CLOCKWISE direction mein move karo.
 *    Jo FIRST SERVER milega, wahi key ka owner hoga.
 *
 *    Golden Rule: Key -> First server clockwise
 */

/**
 * 10. KEY LOOKUP EXAMPLE
 *     Servers: S0 = 100, S1 = 300, S2 = 500, S3 = 700, S4 = 900
 *
 *     KEY 1: position 250 -> clockwise S1(300)  => S1
 *     KEY 2: position 620 -> clockwise S3(700)  => S3
 *     KEY 3: position 950 -> ring end aa gaya: 950 -> 999 -> 0
 *            next server S0(100)                => S0
 *
 *     This is called WRAP AROUND.
 */

/**
 * 11. KEY EXACTLY SERVER POSITION PAR HO
 *     Key position = 700, S3 position = 700 -> Key -> S3.
 *     Rule: exact position match ho to wahi server owner hai.
 */

/**
 * 12. SERVER ADD KARNE KA ADVANTAGE
 *     Existing ring: S0 = 100, S1 = 300, S2 = 500, S3 = 700, S4 = 900
 *     New server   : S5 = 400
 *
 *     Before: S1(300) -----> S2(500)
 *     After : S1(300) -> S5(400) -> S2(500)
 *
 *     S2 pehle 300 -> 500 range own karta tha.
 *     Ab S5 own karega 300 -> 400, S2 own karega 400 -> 500.
 *
 *     Sirf 300 -> 400 range ki keys move hongi.
 *     Baaki keys same server par rahengi.
 */

/**
 * 13. SERVER ADD EXAMPLE
 *     Key positions: 250, 350, 450, 620, 950
 *
 *     Before S5:  250 -> S1, 350 -> S2, 450 -> S2, 620 -> S3, 950 -> S0
 *     After S5 (at 400):
 *                 250 -> S1
 *                 350 -> S5   <-- MOVED
 *                 450 -> S2
 *                 620 -> S3
 *                 950 -> S0
 *
 *     Sirf 350 wali key move hui.
 *     THIS IS THE MAIN ADVANTAGE OF CONSISTENT HASHING.
 */

/**
 * 14. SERVER REMOVE KARNE KA ADVANTAGE
 *     S4 = 900 fail ho gaya. S4 ka range: 700 -> 900.
 *     Remove hone ke baad keys clockwise next server (S0 = 100) par jayengi.
 *
 *     key = 850: Before -> S4, After S4 failure -> S0.
 *     key = 620: still -> S3.
 *
 *     Sirf failed server ki range move hoti hai.
 */

/**
 * 15. KITNI KEYS MOVE HOTI HAIN?
 *     N balanced servers + 1 new server -> approximately 1 / (N + 1) keys move.
 *
 *     Example: 5 -> 6 servers -> 1 / 6 = ~16.7%
 *
 *     IMPORTANT: ye approximate average hai. Exact percentage depend karega:
 *     server positions, key distribution, hash distribution.
 */

/**
 * 16. IMPORTANT: CONSISTENT HASHING DATA KO AUTOMATICALLY MOVE NAHI KARTA
 *     Interview mein important distinction.
 *
 *     Consistent Hashing ownership / routing decide karta hai.
 *     It does NOT automatically: transfer data, copy data,
 *     repair replicas, decide durable writes.
 *
 *     Example: S4 removed. Consistent hashing kehta hai
 *     "Ab ye key S0 ki responsibility hai."
 *     Lekin actual data S4 -> S0 transfer karna hai ya nahi,
 *     woh separate mechanism ki responsibility hai.
 *
 *     Cache      : data missing ho to database se dobara load ho sakta hai.
 *     Persistent : proper data migration required hai.
 */

/**
 * 17. PROBLEM: UNEVEN RANGES
 *     Real world mein server names hash karne par positions perfectly
 *     evenly distribute nahi hoti.
 *
 *     Example: S1 -> small range (less keys), S2 -> huge range (more keys)
 *     S2 par: more traffic, more memory, more load.
 *
 *     So simple one-position-per-server perfect balancing guarantee nahi karta.
 */

/**
 * 18. SERVER FAILURE + UNEVEN RANGE
 *     S2 ka range bahut large hai. S2 fails -> uska complete range
 *     next server ko mil jayega -> neighbor server suddenly overloaded.
 *
 *     Solution: VIRTUAL NODES
 */

/**
 * 19. VIRTUAL NODES / VNODES
 *     Ek physical server ko ring par multiple positions dena.
 *
 *     Instead of S1, we create S1-0, S1-1, S1-2.
 *     All of these point to Physical Server S1.
 *
 *       Physical Server S1 --+-- S1-0 -> position 10
 *                            +-- S1-1 -> position 70
 *                            +-- S1-2 -> position 120
 *
 *     Similarly: S2-0, S2-1, S2-2.
 */

/**
 * 20. VNODE LOOKUP
 *     Key position = 75
 *     Ring: S1-0 = 10, S3-0 = 30, S2-0 = 50, S1-1 = 70,
 *           S2-1 = 80, S3-1 = 90, S1-2 = 120
 *
 *     First clockwise vnode: S2-1 at 80 -> key 75 -> S2.
 *     S2-1 virtual node hai, but actual request -> Physical Server S2.
 */

/**
 * 21. WHY VIRTUAL NODES HELP?
 *     WITHOUT vnodes: one server -> one large range.
 *     WITH vnodes   : one server -> many small ranges.
 *
 *     Benefits:
 *       1. Better load distribution
 *       2. Failure load multiple servers mein spread
 *       3. Different server capacities handle kar sakte hain
 */

/**
 * 22. FAILURE WITH VIRTUAL NODES
 *     S2 has S2-0, S2-1, S2-2. S2 fails.
 *
 *     Instead of entire S2 range -> one neighbor, we get:
 *       S2-0 range -> S1
 *       S2-1 range -> S3
 *       S2-2 range -> S1
 *
 *     Load multiple servers mein spread ho jata hai.
 */

/**
 * 23. DIFFERENT SERVER CAPACITY
 *     S1 = weak server, S2 = powerful server.
 *     Assign: S1 -> 50 virtual nodes, S2 -> 100 virtual nodes.
 *     S2 ko more expected share milega.
 *
 *     This is called capacity weighting.
 */

/**
 * 24. VIRTUAL NODE TRADE-OFF
 *     More virtual nodes:
 *       + Better distribution
 *       + Better failure spreading
 *       + Better capacity weighting
 *
 *     BUT:
 *       - More metadata
 *       - More routing information
 *       - Membership changes par more work
 *
 *     Infinite vnodes nahi banate.
 *     Practical approach: enough vnodes + measure actual distribution.
 *
 *     Tens se few hundred virtual nodes per server ek common starting point
 *     ho sakta hai, lekin exact number system ke distribution ko measure
 *     karke choose karna chahiye.
 */

/**
 * 25. CONSISTENT HASH RING IMPLEMENTATION
 *     Actual code mein ring ko circular data structure banana zaroori nahi.
 *
 *     Maintain:
 *       Sorted Array: [10, 30, 50, 70, 80, 90, 120]
 *       Mapping (position -> physical server):
 *         10 -> S1,  30 -> S3,  50 -> S2,  70 -> S1,
 *         80 -> S2,  90 -> S3, 120 -> S1
 */

/**
 * 26. KEY LOOKUP USING BINARY SEARCH
 *     keyHash = 75, sorted positions [10, 30, 50, 70, 80, 90, 120].
 *     Find first position >= 75 -> 80 -> server at position 80.
 *
 *     Complexity: O(log V), where V = number of virtual nodes.
 */

/**
 * 27. WRAP AROUND IN IMPLEMENTATION
 *     keyHash = 150, largest position = 120 -> no position >= 150.
 *     Wrap around: use first position (10) -> 150 -> server at position 10.
 */

/**
 * 28. EMPTY RING
 *     Ring empty hai -> no server available -> return Error.
 */

/**
 * 29. HASH COLLISION
 *     Kabhi-kabhi two virtual nodes same hash position par aa sakte hain.
 *     Example: S1-2 -> 500, S2-4 -> 500.
 *
 *     Production implementation mein deterministic rule hona chahiye,
 *     e.g. server ID sorting ke basis par ek fixed winner choose karna.
 *     Goal: every client same result dekhe.
 */

/**
 * 30. HASH FUNCTION
 *     Production ke liye: fast, stable, well-distributed, non-cryptographic hash.
 *     Examples: MurmurHash, xxHash, CityHash.
 *
 *     Example implementation mein FNV-1a use hota hai, kyunki simple hai
 *     aur languages mein consistent output deta hai.
 *
 *     Important: hash function ka output stable hona chahiye.
 */

/**
 * 31. SAME HASH FUNCTION + SAME SERVER LIST
 *     Every client ke paas hona chahiye: same server list + same hash function.
 *
 *     Otherwise:
 *       Client A: user:42 -> S2
 *       Client B: user:42 -> S3
 *     Consistency break ho jayegi.
 *
 *     Usually a configuration service ring updates publish karti hai,
 *     taaki all clients eventually same view par converge karein.
 */

/**
 * 32. REPLICATION WITH CONSISTENT HASHING
 *     Suppose 3 copies chahiye.
 *     First: primary owner. Then clockwise move karo: next DISTINCT physical server.
 *
 *     Example: Key = 75
 *       position 80 -> S2  => Primary
 *       position 90 -> S3  => Replica 1
 *       position 120 -> S1 => Replica 2
 *
 *     Final: S2 Primary, S3 Replica, S1 Replica.
 */

/**
 * 33. WHY "DISTINCT PHYSICAL SERVER" IMPORTANT?
 *     Primary: S2-1, next vnode: S2-2 — dono same physical machine S2 ke hain.
 *     S2 crashes -> primary + replica dono lost.
 *
 *     Therefore replica select karte time physical server distinct hona chahiye.
 *     Production mein replicas ko different servers, racks,
 *     availability zones mein spread karna useful hai.
 */

/**
 * 34. RING DOES NOT HANDLE REPLICATION ITSELF
 *     Ring sirf decide karta hai: primary kahan, replica candidates kahan.
 *
 *     Other components handle: data copying, write success rules,
 *     replica repair, missing replica recovery.
 */

/**
 * 35. WHEN TO USE CONSISTENT HASHING?
 *     Good use cases:
 *       1. Distributed Cache
 *       2. Storage Sharding
 *       3. Stateful Routing
 *
 *     Examples:
 *       userId -> same cache server
 *       productId -> same shard
 *       sessionId -> same server
 */

/**
 * 36. WHEN NOT TO USE IT?
 *     Agar any server can handle any request, normal load balancer simpler hai.
 *
 *       Client --> LB --> S1 / S2 / S3
 *
 *     Agar servers stateless hain, request kisi bhi healthy server ko ja sakti hai.
 *     Us case mein consistent hashing ki complexity unnecessary hai.
 */

/**
 * 37. CONSISTENT HASHING vs LOAD BALANCING
 *     CONSISTENT HASHING: key-based routing; same key usually same owner.
 *     LOAD BALANCER     : healthy server choose karta hai; any server can handle.
 *
 *     Stateful / key affinity -> Consistent Hashing
 *     Stateless service       -> Normal Load Balancer
 */

/**
 * 38. OTHER APPROACHES
 *     Consistent hashing hi only solution nahi hai.
 *
 *     1. RENDEZVOUS HASHING
 *        Har server ko key ke liye score do; highest score wala owner banega.
 *        Good for: small / medium server sets.
 *
 *     2. FIXED PARTITIONS
 *        Keys ko fixed number of partitions mein divide karo (e.g. 64).
 *        key -> hash(key) % 64 -> Partition 17 -> S3
 *        Partition count fixed hai; server mapping change ho sakti hai,
 *        but partition count same rehta hai.
 *        Good for: logs, queues, databases.
 */

/**
 * 39. CONSISTENT HASHING vs MODULO HASHING
 *
 *     MODULO: server = hash(key) % N
 *       N change hua -> many keys remapped.
 *
 *     CONSISTENT HASHING: Key + Servers -> Hash Ring
 *       Server add/remove -> only nearby range affected.
 */

/**
 * 40. INTERVIEW QUESTIONS
 *
 * Q1. What is consistent hashing?
 * A : Ek technique jo keys ko servers ke beech map karti hai while minimizing
 *     key movement when servers join or leave the cluster.
 *
 * Q2. Why do we need consistent hashing?
 * A : Modulo hashing mein server count change hone par large number of keys
 *     remap ho jati hain. Consistent hashing mein sirf affected range ki keys
 *     generally move hoti hain.
 *
 * Q3. What is the problem with hash(key) % numberOfServers?
 * A : Server count change hote hi modulo value change ho jati hai, isliye
 *     large number of keys ka owner change ho sakta hai.
 *
 * Q4. What is a hash ring?
 * A : Ek fixed circular hash space jahan servers aur keys dono ko hash karke
 *     positions assign ki jati hain. Key ka owner first server clockwise hota hai.
 *
 * Q5. How is a key mapped to a server?
 * A : 1. Key ko hash karo.  2. Hash ko ring par position karo.
 *     3. Clockwise move karo.  4. First server position ko owner banao.
 *
 * Q6. What happens when a new server joins?
 * A : New server ring par ek ya multiple positions leta hai. Sirf uski newly
 *     acquired range ki keys previous owner se new server par remap hoti hain.
 *
 * Q7. What happens when a server leaves?
 * A : Us server ki owned keys clockwise next available servers ko remap hoti
 *     hain. Baaki keys normally unaffected rehti hain.
 *
 * Q8. What are virtual nodes?
 * A : Ek physical server ko ring par multiple positions dena
 *     (S1 -> S1-0, S1-1, S1-2). Ye load distribution improve karte hain.
 *
 * Q9. Why are virtual nodes needed?
 * A : Without vnodes, server positions uneven ho sakti hain aur ek server
 *     bahut large range own kar sakta hai. Vnodes multiple smaller ranges
 *     banake load better distribute karte hain.
 *
 * Q10. What happens when a vnode server fails?
 * A : Us server ke multiple ranges different neighboring servers ko mil jate
 *     hain, isliye ek single neighbor par huge load nahi aata.
 *
 * Q11. Can consistent hashing guarantee perfectly equal distribution?
 * A : No. Vnodes distribution ko statistically improve karte hain, but
 *     perfect equality guarantee nahi hoti.
 *
 * Q12. Does consistent hashing move data?
 * A : No. Ye mainly ownership/routing decide karta hai.
 *     Actual data migration separate process hai.
 *
 * Q13. How does it support replication?
 * A : Primary owner find karne ke baad clockwise continue karke additional
 *     DISTINCT physical servers replicas ke liye choose kiye ja sakte hain.
 *
 * Q14. Why should replicas be on different physical servers?
 * A : Agar primary aur replica same machine par hain aur machine fail ho
 *     gayi, dono copies lost ho sakti hain.
 *
 * Q15. When would you prefer a normal load balancer?
 * A : Jab service stateless ho aur koi bhi healthy server request handle kar
 *     sake. Us case mein consistent hashing ki complexity unnecessary hai.
 *
 * Q16. What is the lookup complexity?
 * A : Sorted virtual-node positions ke saath binary search: O(log V),
 *     where V = number of virtual nodes.
 *
 * Q17. What must all clients agree on?
 * A : Same hash function + same server/ring membership. Otherwise same key
 *     different servers par route ho sakti hai.
 */

/**
 * 41. REAL SYSTEM DESIGN EXAMPLE
 *     Distributed Redis-like cache, servers S1, S2, S3. User profile: user:1001.
 *
 *     Request -> hash(user:1001) -> Hash Ring -> First Server Clockwise -> S2
 *     -> Cache lookup
 *
 *     Same user request again -> S2. Cache locality maintained.
 */

/**
 * 42. IF S4 IS ADDED
 *     Existing: S1, S2, S3. Add: S4.
 *     Ring changes only around S4's positions.
 *     Most existing keys remain on same servers.
 *
 *     Benefit: better cache hit rate, less data reshuffling, less load spike.
 */

/**
 * 43. IF S2 FAILS
 *     Consistent hashing S2 ki ranges identify karta hai.
 *     Those ranges -> next available owners.
 *     With virtual nodes: load multiple servers mein spread ho sakta hai.
 */

/**
 * 44. MOST IMPORTANT THINGS TO REMEMBER
 *     1. Modulo hashing: hash(key) % N. N changes -> many keys move.
 *     2. Consistent hashing: Key + Servers -> Hash Ring.
 *     3. Key owner: first server clockwise.
 *     4. Server added  : only affected range moves.
 *     5. Server removed: only its owned range moves.
 *     6. Virtual nodes : better distribution, better failure handling,
 *                        capacity weighting.
 *     7. Consistent hashing decides placement. It does NOT automatically
 *        move data, copy replicas, repair replicas, or decide durable writes.
 */

/**
 * 45. EASY MEMORY TRICK
 *     MODULO HASHING    : "Server count change karo, mapping almost
 *                          completely change ho sakti hai."
 *     CONSISTENT HASHING: "Ring par nearby ownership change karo."
 *     VIRTUAL NODES     : "Ek server ko ring par multiple small positions do."
 */

/**
 * ONE-LINE INTERVIEW ANSWER
 *   "Consistent hashing maps both keys and servers onto a fixed hash ring and
 *    assigns each key to the first server clockwise, so when servers are added
 *    or removed, only a small portion of keys need to be remapped instead of
 *    reshuffling the entire cluster."
 */